# Phase 1 — SimCLR reference encoder

Trains the Phase-1 SimCLR reference encoder (2 seeds) + the supervised-reference ceiling on Shapes3D, then runs the encoder-quality gate (SimCLR vs random-encoder vs supervised).

The dataset and all checkpoints/results are written to **Google Drive**, so they persist across sessions. Training checkpoints every epoch and auto-resumes — if a training cell stops partway, just **re-run that same cell**.

**Before running:** Runtime → Change runtime type → **GPU**.

In [ ]:
!nvidia-smi

## 1. Mount Drive (persistent dataset + checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the repo + redirect dataset/results to Drive

The repo lives on fast local disk (`/content`); only `data/raw` (the 255 MB dataset) and `results` (checkpoints + gate output) are symlinked to Drive so they survive session restarts.

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/content/probe-capacity-invariance"
DRIVE_DIR = "/content/drive/MyDrive/probe-capacity-invariance-data"

os.makedirs(f"{DRIVE_DIR}/data/raw", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

!rm -rf {REPO_DIR}/data/raw {REPO_DIR}/results
!ln -s {DRIVE_DIR}/data/raw {REPO_DIR}/data/raw
!ln -s {DRIVE_DIR}/results {REPO_DIR}/results

%cd {REPO_DIR}

## 3. Install dependencies

Install the package without disturbing the preinstalled, CUDA-matched `torch`/`torchvision`; add the one extra dependency (`h5py`).

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Runtime > Change runtime type > GPU")

## 4. Download Shapes3D (cached to Drive — runs once across sessions)

In [ ]:
!python -m src.data.shapes3d --download

## 5. Train the reference encoder — seed 0

Checkpoints every epoch to Drive. If this cell stops, re-run it to resume from the last saved epoch.

In [ ]:
!python -m src.encoders.train_simclr --config configs/run/reference_cuda.yaml --set run.seed=0

## 6. Train the reference encoder — seed 1 (seed-to-seed reproducibility check)

In [ ]:
!python -m src.encoders.train_simclr --config configs/run/reference_cuda.yaml --set run.seed=1

## 7. Train the supervised-reference encoder (recoverability ceiling)

In [ ]:
!python -m src.encoders.supervised --config configs/run/supervised.yaml --set run.seed=0

## 8. Run the encoder-quality gate

Per-factor normalized recoverability for SimCLR vs random-encoder vs supervised-reference → PASS/FAIL against the Phase-1 thresholds.

In [ ]:
!python -m src.eval.quality_gate \
    --config configs/run/reference_cuda.yaml \
    --simclr results/encoders/standard_strong_seed0/backbone.pt \
             results/encoders/standard_strong_seed1/backbone.pt \
    --supervised results/encoders/supervised_seed0/backbone.pt \
    --random-seed 0 \
    --out results/quality_gate/reference.json

## 9. Print the gate report

In [ ]:
import json
print(json.dumps(json.load(open("results/quality_gate/reference.json")), indent=2))